In [ ]:
import matplotlib
import tensorflow
import numpy 
# scenario 1 - lancio del razzo con calcolo metabolismo basale standard secondo le convenzioni di scienze della nutrizione
def calcola_calorie_lancio_integrato(heart_rate, peso_kg, durata_minuti, alarm_triggered, score):
    consumo_base_minuto = (peso_kg * 1.0) / 60  #standard in base alle forze G riscontrabili in atm
    # 1. Moltiplicatore di attività basato sulla frequenza cardiaca
    if heart_rate > 140:
        moltiplicatore_attivita = 3.5  # Iperventilazione e contrasto Forze G
    elif heart_rate > 100:
        moltiplicatore_attivita = 2.0
    else:
        moltiplicatore_attivita = 1.2
        # 2. Amplificatore dello Stress Biometrico basato sulle variabili del file ECG
    moltiplicatore_stress_ecg = 1.0
    if alarm_triggered:
        moltiplicatore_stress_ecg += 0.25  # Costo metabolico dell'aritmia
    moltiplicatore_stress_ecg += (score * 2.0)  # Peso dinamico dell'errore di ricostruzione MAE

    totalcal= consumo_base_minuto*moltiplicatore_attivita*moltiplicatore_stress_ecg*durata_minuti
    return totalcal


In [ ]:
#scenario n2- a bordo iss-"statica"

def calcola_calorie_stazione_integrato(heart_rate, peso_kg, durata_minuti, ecg_is_anomaly, ecg_mae_score):
    # In microgravità il consumo basale a riposo si riduce del 10% rispetto alla Terra
    consumo_basale_microgravita = (peso_kg * 0.9) / 60
    
    # 1. Calcolo della spesa energetica in base all'attività (riposo vs macchinari ARED/T2)
    if heart_rate > 120:
        # Fase di allenamento intensivo contro l'atrofia muscolare
        calorie_minuto = (heart_rate * 0.05) + (peso_kg * 0.02)
    else:
        # Lavoro scientifico o riposo fluttuando nella stazione
        calorie_minuto = consumo_basale_microgravita * 1.1
        
    # 2. Amplificatore dello Stress Biometrico basato dall'Autoencoder ECG
    moltiplicatore_stress_ecg = 1.0
    if ecg_is_anomaly:
        moltiplicatore_stress_ecg += 0.20
    moltiplicatore_stress_ecg += (ecg_mae_score * 1.5)

    calorie_totali = calorie_minuto * durata_minuti * moltiplicatore_stress_ecg
    return round(max(0, calorie_totali), 2)

In [ ]:
#scenario n3 - a bordo ISS- esercizio fisico obbligatorio- l esercizio fisico, presi in considerazione tre esercizi.
#ARED- esercizio a pistoni che ricorda il sollevamento pesi (non fattibile a 0g)
#T2- il classico ibrido tapis roulant
#CEVIS- una cyclette x 0g


def calcola_calorie_allenamento_iss(heart_rate, peso_kg, durable_minuti, macchinario, alarm_triggered, score):
    """
    Scenario Speciale: Contromisure ginniche quotidiane obbligatorie a bordo della ISS.
    Calcola il consumo energetico specifico per l'uso di CEVIS, T2 e ARED.
    """
    macchinario_clean = macchinario.upper()
    
    if macchinario_clean not in ['CEVIS', 'T2', 'ARED']:
        raise ValueError("Macchinario non valido. Selezionare tra 'CEVIS', 'T2' o 'ARED'.")

    # 1. Costo metabolico basale al minuto specifico per ogni macchinario NASA
    if macchinario_clean == 'T2':
        # Tapis Roulant: sforzo elevato dovuto ai vincoli elastici di ritenzione del peso
        met_stimato = (heart_rate * 0.055) + (peso_kg * 0.025)
    elif macchinario_clean == 'ARED':
        # Advanced Resistive Exercise Device: lavoro di forza pura contro la perdita ossea
        met_stimato = (heart_rate * 0.045) + (peso_kg * 0.035)
    elif macchinario_clean == 'CEVIS':
        # Cycle Ergometer: lavoro cardiovascolare ciclistico in assenza di gravità
        met_stimato = (heart_rate * 0.040) + (peso_kg * 0.020)

    # 2. Amplificatore dello Stress Biometrico derivato dall'Autoencoder ECG
    moltiplicatore_stress_ecg = 1.0
    if alarm_triggered:
        moltiplicatore_stress_ecg += 0.25  # Maggior dispendio dovuto all'inefficienza cardiaca in aritmia
    moltiplicatore_stress_ecg += (score * 2.0)  # Impatto dinamico del MAE score

    calorie_totali = met_stimato * durable_minuti * moltiplicatore_stress_ecg
    return round(max(0, calorie_totali), 2)

In [ ]:
# scenario nel caso di uscita in Rover/ tuta EVA
def calcola_calorie_eva_rover_integrato(heart_rate, peso_kg, durata_minuti, oxygen_perc, alarm_triggered, score):
    # Spesa energetica base elevata dovuta alla pressurizzazione della tuta spaziale
    calorie_minuto_base = (heart_rate * 0.06) + (peso_kg * 0.03)
    
    # 1. Moltiplicatore ambientale legato all'ipossia e alla miscela di ossigeno erogata
    if oxygen_perc < 90:
        moltiplicatore_ambiente = 1.30  # +30% di sforzo per debito di ossigeno e iperventilazione
    elif oxygen_perc < 95:
        moltiplicatore_ambiente = 1.15
    else:
        moltiplicatore_ambiente = 1.0
        
    # 2. Amplificatore dello Stress Biometrico basato sulle variabili del file ECG
    moltiplicatore_stress_ecg = 1.0
    if alarm_triggered:
        moltiplicatore_stress_ecg += 0.30  # Forte penalità metabolica in ambiente ostile
    moltiplicatore_stress_ecg += (score * 2.5)  # Massimo impatto del punteggio MAE sulla reazione di panico

    calorie_totali = calorie_minuto_base * durata_minuti * moltiplicatore_ambiente * moltiplicatore_stress_ecg
    return round(max(0, calorie_totali), 2)

In [ ]:
#modulo per nutrizione smart in base al numero di cal perse e liquidi, una corretta integrazione è fondamentale per prevenire la catabolisi muscolare e la perdita di massa ossea
def reintegration(scenario, durataminuti, heart_rate, peso_kg, consumo_basale_microgravita, sex, totale_kcal_giornaliere, totale_acqua_giornaliera):
    scenario_original = scenario.upper()
    if scenario_original == "STATION":
        if heart_rate > 120:
            calorie_minuto = (heart_rate * 0.05) + (peso_kg * 0.02)
            acquaminuto = calorie_minuto
        else:
            calorie_minuto = consumo_basale_microgravita * 1.1
            acquaminuto = calorie_minuto
    elif scenario_original == "OUTSIDE":
        if heart_rate > 120:
            calorie_minuto = (heart_rate * 0.05) + (peso_kg * 0.02)
            acquaminuto = calorie_minuto + 20 * calorie_minuto / 100   #riduzione del 20% secondo standard NASA nello spazio
        else:
            calorie_minuto = consumo_basale_microgravita * 1.1
            acquaminuto = calorie_minuto + 20 * calorie_minuto / 100
    else:
        calorie_minuto = consumo_basale_microgravita * 1.1
        acquaminuto = calorie_minuto

    # --- BLOCCO PASTO ---
    if calorie_minuto * durataminuti <= 150:
        tipo_pasto = "light snack"
    elif 150 < calorie_minuto * durataminuti <= 400:
        tipo_pasto = "high carb-protein based snack"
    else:
        tipo_pasto = "URGENT INTEGRATION NEEDED"

    # --- BLOCCO IDRATAZIONE ---
    if acquaminuto * durataminuti <= 100:
        hydration = "minimal fluid loss, standard sips"
    elif 100 < acquaminuto * durataminuti <= 150:
        hydration = "standard hydration with small isotonic solution"
    elif 150 < acquaminuto * durataminuti < 300:
        hydration = "uncommon liquid loss, if you didn't engage in significant physical activity significantly increase your integration"
    elif 300 <= acquaminuto * durataminuti <= 600:
        hydration = "Significant loss of liquids detected - compatible with physical exercise or EVA exploration - significant hydration and salt reintegration is suggested"
    else:
        hydration = "CRITICAL FLUID LOSS (>600 mL) - urgent electrolyte reintegration required"

    # --- BILANCIO CALORICO GIORNALIERO ---
    if sex.upper() == "M" and 2300 < totale_kcal_giornaliere <= 3000:     #aumento di 100 kcal rispetto agli standard in caso di necessità di extra dispendio energetico
        cal_balance = "optimal kcal intake"
    elif sex.upper() == "F" and 1900 < totale_kcal_giornaliere <= 2400:
        cal_balance = "optimal kcal intake"
    else:
        cal_balance = "kcal supplement required"

    # --- BILANCIO IDRICO GIORNALIERO ---
    if sex.upper() == "M" and 2300 < totale_acqua_giornaliera <= 2300:     
        hydro_balance = "adequate hydration was provided during the day"
    elif sex.upper() == "F" and 1900 < totale_acqua_giornaliera <= 2100:
        hydro_balance = "adequate hydration was provided during the day"
    else:
        hydro_balance = "inadequate fluid integration, hydration necessary"

    return calorie_minuto * durataminuti, acquaminuto * durataminuti, tipo_pasto, hydration, cal_balance, hydro_balance

#  REGISTRO ATTIVITA' GIORNALIERE 
diario_giornaliero = []

def registra_attivita(nome_attivita, calorie_bruciate, liquidi_persi):
    diario_giornaliero.append({                             #registro che aggiunge attività giornaliere per stimare il bilancio calorico
        "attivita": nome_attivita,
        "kcal": calorie_bruciate,
        "ml_acqua": liquidi_persi
    })
    totale_kcal = sum(item["kcal"] for item in diario_giornaliero)             #somma elementi per il bilancio
    totale_acqua = sum(item["ml_acqua"] for item in diario_giornaliero)
    return totale_kcal, totale_acqua

# ESEMPIO
registra_attivita("Sonno e riposo (8h)", 580, 580)
registra_attivita("Lavoro scientifico in cabina (10h)", 850, 850)
tot_kcal, tot_acqua = registra_attivita("Allenamento ARED + T2 (2h)", 650, 750)

print(reintegration("STATION", 120, 145, 75, 1.12, "M", tot_kcal, tot_acqua))
    